# Principles of 3D Reconstruction: Structure-from-Motion and Multi-View Stereo

This workflow explores the fundamentals of digital 3D reconstruction using COLMAP. In the context of computer vision, we aim to recover the three-dimensional structure of a scene from a collection of two-dimensional images.

### The Reconstruction Pipeline
1.  **Feature Extraction & Matching**: Identifying unique points in every image and finding where those points appear across the entire dataset.
2.  **Structure-from-Motion (SfM)**: Solving for the position of the camera in space (extrinsic parameters) and the sparse geometry of the scene (point clouds).
3.  **Dense Reconstruction (MVS)**: Using the solved camera positions to find depth information for every pixel, creating a highly detailed surface model.
4.  **Surface Meshing**: Connecting the dense points into a continuous geometric mesh for use in 3D modeling software.

### Hardware Considerations
3D reconstruction is computationally intensive. We utilize GPU acceleration (via CUDA) specifically for the **Patch Match Stereo** phase, which involves millions of depth comparisons to generate a dense model.

## First let's make sure our instance is connected to a GPU

In [29]:
import torch
print("CUDA available:", torch.cuda.is_available())

CUDA available: True


### Environment Preparation
To ensure consistent results across different hardware types, we compile the reconstruction engine from source. This ensures all modules, including the GPU-accelerated stereo matching and Poisson meshing tools, are properly linked to the system libraries.

# Install COLMAP from Ubuntu packages

In [30]:

!sudo apt update
!sudo apt-get install -y colmap xvfb
!which colmap
!colmap -h | head -n 20

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 2s (2,040 B/s)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
56 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of

### 🛠️ Step 1: Build COLMAP from source with CUDA
We need to uninstall the CPU version and build a new one to enable the GPU for Dense Reconstruction.

In [31]:
# Remove the existing CPU-only version
!sudo apt-get remove -y colmap

# Install all dependencies required for building COLMAP 3.9.1 with GPU support
!sudo apt-get update
!sudo apt-get install -y cmake libboost-program-options-dev libboost-filesystem-dev \
    libboost-graph-dev libboost-system-dev libboost-test-dev libsuitesparse-dev \
    libfreeimage-dev libgoogle-glog-dev libgflags-dev libglew-dev \
    libqt5opengl5-dev qtbase5-dev libceres-dev libflann-dev libcgal-dev libsqlite3-dev libmetis-dev

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following packages will be REMOVED:
  colmap
0 upgraded, 0 newly installed, 1 to remove and 56 not upgraded.
After this operation, 9,647 kB disk space will be freed.
(Reading database ... 131999 files and directories currently installed.)
Removing colmap (3.7-2) ...
Processing triggers for mailcap (3.70+nmu1ubuntu1) ...
Processing triggers for man-db (2.10.2-1) ...
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy In

In [32]:
# Clone and build COLMAP from source with CUDA enabled
%cd /content
!rm -rf colmap
!git clone https://github.com/colmap/colmap.git
%cd colmap
!git checkout 3.9.1
!mkdir build
%cd build
# Added -DCMAKE_CUDA_ARCHITECTURES=native to fix the configuration error
!cmake .. -DGUI_ENABLED=OFF -DCUDA_ENABLED=ON -DCMAKE_CUDA_ARCHITECTURES=native
!make -j$(nproc)
!sudo make install

# Verify that CUDA is now detected in the output
!colmap -h | head -n 20

/content
Cloning into 'colmap'...
remote: Enumerating objects: 42012, done.
remote: Counting objects: 100% (1267/1267), done.
remote: Compressing objects: 100% (648/648), done.
remote: Total 42012 (delta 976), reused 624 (delta 619), pack-reused 40745 (from 3)
Receiving objects: 100% (42012/42012), 78.07 MiB | 18.40 MiB/s, done.
Resolving deltas: 100% (32798/32798), done.
/content/colmap
Note: switching to '3.9.1'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at e9903641 Patch release 3.9.1 (#2335)
/content/col

In [33]:
!sudo apt-get update
!sudo apt-get install -y libfreeimage3 libfreeimage-dev
# Use the absolute path to our custom-built binary
COLMAP_BIN = '/usr/local/bin/colmap'
!$COLMAP_BIN feature_extractor -h | grep -i "image"

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 2s (1,917 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (

# Skip source build

In [34]:
%cd /content
!echo "Using the Ubuntu-packaged COLMAP binary; no Git clone or CMake build is needed."

/content
Using the Ubuntu-packaged COLMAP binary; no Git clone or CMake build is needed.


# Verify COLMAP installation


In [35]:
COLMAP_BIN = '/usr/local/bin/colmap'
!ls -l $COLMAP_BIN
!$COLMAP_BIN -h | head -n 20

-rwxr-xr-x 1 root root 17540248 May 18 17:00 /usr/local/bin/colmap
COLMAP 3.9.1 -- Structure-from-Motion and Multi-View Stereo
(Commit e9903641 on 2024-01-08 with CUDA)

Usage:
  colmap [command] [options]

Documentation:
  https://colmap.github.io/

Example usage:
  colmap help [ -h, --help ]
  colmap gui
  colmap gui -h [ --help ]
  colmap automatic_reconstructor -h [ --help ]
  colmap automatic_reconstructor --image_path IMAGES --workspace_path WORKSPACE
  colmap feature_extractor --image_path IMAGES --database_path DATABASE
  colmap exhaustive_matcher --database_path DATABASE
  colmap mapper --image_path IMAGES --database_path DATABASE --output_path MODEL
  ...



## Sparse Reconstruction (Structure-from-Motion)

Before running these steps, ensure your source images are placed in the `frames` directory.

In this phase, we perform:
- **Feature Extraction**: Detecting SIFT features that remain consistent across changes in scale and rotation.
- **Exhaustive Matching**: Comparing every image against every other image to build a map of correspondences.
- **Mapping**: Triangulating the matches to determine the 3D coordinates of the points and the poses of the cameras.

In [36]:
COLMAP_BIN = '/usr/local/bin/colmap'
IMAGES_PATH = '/content/frames'
DATABASE_PATH = '/content/database.db'
WORKSPACE_PATH = '/content'

# 1. Feature Extraction
!$COLMAP_BIN feature_extractor \
    --database_path $DATABASE_PATH \
    --image_path $IMAGES_PATH \
    --ImageReader.camera_model SIMPLE_RADIAL \
    --SiftExtraction.use_gpu 0

# 2. Exhaustive Matching
!$COLMAP_BIN exhaustive_matcher \
    --database_path $DATABASE_PATH \
    --SiftMatching.use_gpu 0

# 3. Mapper (Sparse Reconstruction)
!mkdir -p $WORKSPACE_PATH/sparse
!$COLMAP_BIN mapper \
    --database_path $DATABASE_PATH \
    --image_path $IMAGES_PATH \
    --output_path $WORKSPACE_PATH/sparse

W0518 17:00:41.714324 44560 feature_extraction.cc:403] Your current options use the maximum number of threads on the machine to extract features. Extracting SIFT features on the CPU can consume a lot of RAM per thread for large images. Consider reducing the maximum image size and/or the first octave or manually limit the number of extraction threads. Ignore this warning, if your machine has sufficient memory for the current settings.
I0518 17:00:41.714565 44562 misc.cc:198] 
Feature extraction
I0518 17:00:41.874037 44567 feature_extraction.cc:254] Processed file [1/60]
I0518 17:00:41.874100 44567 feature_extraction.cc:257]   Name:            frame_0001.png
E0518 17:00:41.874106 44567 feature_extraction.cc:263] Failed to read image file format.
I0518 17:00:41.975656 44567 feature_extraction.cc:254] Processed file [2/60]
I0518 17:00:41.975690 44567 feature_extraction.cc:257]   Name:            frame_0002.png
E0518 17:00:41.975699 44567 feature_extraction.cc:263] Failed to read image file

In [37]:
import os
from PIL import Image

# Troubleshooting: Check if images are valid and readable by Python
images = [f for f in os.listdir(IMAGES_PATH) if f.endswith(('.png', '.jpg', '.jpeg'))]
print(f"Found {len(images)} images in {IMAGES_PATH}")

for img_name in images[:5]: # Check first 5
    try:
        img_path = os.path.join(IMAGES_PATH, img_name)
        with Image.open(img_path) as img:
            print(f"OK: {img_name} - {img.format} {img.size}")
    except Exception as e:
        print(f"CORRUPT or UNREADABLE: {img_name} - {e}")

Found 60 images in /content/frames
OK: frame_0014.png - PNG (1920, 1080)
OK: frame_0021.png - PNG (1920, 1080)
OK: frame_0057.png - PNG (1920, 1080)
OK: frame_0016.png - PNG (1920, 1080)
OK: frame_0012.png - PNG (1920, 1080)


In [38]:
import os
from PIL import Image

# Create a new directory for JPG versions
JPG_IMAGES_PATH = '/content/frames_jpg'
os.makedirs(JPG_IMAGES_PATH, exist_ok=True)

print("Converting PNGs to JPG for better COLMAP compatibility...")
for img_name in os.listdir(IMAGES_PATH):
    if img_name.lower().endswith('.png'):
        try:
            img_path = os.path.join(IMAGES_PATH, img_name)
            with Image.open(img_path) as img:
                # Convert to RGB and save as JPG
                rgb_img = img.convert('RGB')
                target_path = os.path.join(JPG_IMAGES_PATH, img_name.replace('.png', '.jpg'))
                rgb_img.save(target_path, 'JPEG', quality=95)
        except Exception as e:
            print(f"Failed to convert {img_name}: {e}")

print(f"Done! Converted images are in: {JPG_IMAGES_PATH}")

Converting PNGs to JPG for better COLMAP compatibility...
Done! Converted images are in: /content/frames_jpg


In [39]:
# Updated Reconstruction Step using GPU
IMAGES_PATH_JPG = '/content/frames_jpg'
DATABASE_PATH = '/content/database.db'
WORKSPACE_PATH = '/content'
COLMAP_BIN = '/usr/local/bin/colmap'

# Remove old database to start fresh
import os
if os.path.exists(DATABASE_PATH): os.remove(DATABASE_PATH)

# Feature Extraction with GPU
!$COLMAP_BIN feature_extractor \
    --database_path $DATABASE_PATH \
    --image_path $IMAGES_PATH_JPG \
    --ImageReader.camera_model SIMPLE_RADIAL \
    --SiftExtraction.use_gpu 1

# Exhaustive Matching with GPU
!$COLMAP_BIN exhaustive_matcher \
    --database_path $DATABASE_PATH \
    --SiftMatching.use_gpu 1

# Mapper (Sparse Reconstruction)
!mkdir -p $WORKSPACE_PATH/sparse
!$COLMAP_BIN mapper \
    --database_path $DATABASE_PATH \
    --image_path $IMAGES_PATH_JPG \
    --output_path $WORKSPACE_PATH/sparse

I0518 17:01:12.329865 44793 misc.cc:198] 
Feature extraction
I0518 17:01:12.643276 44797 feature_extraction.cc:254] Processed file [1/60]
I0518 17:01:12.643317 44797 feature_extraction.cc:257]   Name:            frame_0001.jpg
I0518 17:01:12.643324 44797 feature_extraction.cc:283]   Dimensions:      1920 x 1080
I0518 17:01:12.643329 44797 feature_extraction.cc:286]   Camera:          #1 - SIMPLE_RADIAL
I0518 17:01:12.643335 44797 feature_extraction.cc:289]   Focal Length:    2304.00px
I0518 17:01:12.643347 44797 feature_extraction.cc:302]   Features:        5793
I0518 17:01:12.700096 44797 feature_extraction.cc:254] Processed file [2/60]
I0518 17:01:12.700129 44797 feature_extraction.cc:257]   Name:            frame_0002.jpg
I0518 17:01:12.700137 44797 feature_extraction.cc:283]   Dimensions:      1920 x 1080
I0518 17:01:12.700145 44797 feature_extraction.cc:286]   Camera:          #2 - SIMPLE_RADIAL
I0518 17:01:12.700152 44797 feature_extraction.cc:289]   Focal Length:    2304.00px
I0

## Dense Reconstruction (Multi-View Stereo)

Sparse reconstruction only provides a 'skeleton' of the scene. To create a full surface, we must:
1.  **Undistort**: Remove lens distortion so that the math of the stereo matching assumes a perfect pinhole camera.
2.  **Patch Match Stereo**: Use the GPU to calculate depth maps for every image by comparing local neighborhoods of pixels.
3.  **Stereo Fusion**: Merge these individual depth maps into a single, unified 3D point cloud.

In [40]:
# Define workspace paths
DENSE_PATH = '/content/dense'
COLMAP_BIN = '/usr/local/bin/colmap'
!mkdir -p $DENSE_PATH

# 1. Undistort images (Required before dense matching)
print("Undistorting images...")
!$COLMAP_BIN image_undistorter \
    --image_path $IMAGES_PATH_JPG \
    --input_path $WORKSPACE_PATH/sparse/0 \
    --output_path $DENSE_PATH \
    --output_type COLMAP \
    --max_image_size 2000

# 2. Patch Match Stereo (The heavy GPU lifting)
print("Running Patch Match Stereo (GPU)...")
!$COLMAP_BIN patch_match_stereo \
    --workspace_path $DENSE_PATH \
    --workspace_format COLMAP \
    --PatchMatchStereo.geom_consistency true

# 3. Stereo Fusion (Creating the final point cloud)
print("Fusing results into a PLY file...")
!$COLMAP_BIN stereo_fusion \
    --workspace_path $DENSE_PATH \
    --workspace_format COLMAP \
    --input_type geometric \
    --output_path $DENSE_PATH/fused.ply

print("Dense reconstruction complete! File is at /content/dense/fused.ply")

Undistorting images...
I0518 17:01:53.275537 45065 misc.cc:198] 
Reading reconstruction
I0518 17:01:53.286150 45065 image.cc:342] => Reconstruction with 14 images and 2648 points
I0518 17:01:53.286358 45067 misc.cc:198] 
Image undistortion
I0518 17:01:53.286908 45067 undistortion.cc:210] Undistorting image [1/14]
I0518 17:01:54.261677 45067 undistortion.cc:210] Undistorting image [2/14]
I0518 17:01:54.382962 45067 undistortion.cc:210] Undistorting image [3/14]
I0518 17:01:55.434048 45067 undistortion.cc:210] Undistorting image [4/14]
I0518 17:01:55.434134 45067 undistortion.cc:210] Undistorting image [5/14]
I0518 17:01:56.257861 45067 undistortion.cc:210] Undistorting image [6/14]
I0518 17:01:56.334739 45067 undistortion.cc:210] Undistorting image [7/14]
I0518 17:01:56.870623 45067 undistortion.cc:210] Undistorting image [8/14]
I0518 17:01:56.936440 45067 undistortion.cc:210] Undistorting image [9/14]
I0518 17:01:57.408370 45067 undistortion.cc:210] Undistorting image [10/14]
I0518 17:

### 🕸️ Step 3: Mesh Generation
Finally, we convert the fused point cloud into a dense surface mesh using Poisson Reconstruction. This produces a `.ply` file that includes mesh faces, making it much easier to view and manipulate in Meshlab.

In [41]:
COLMAP_BIN = '/usr/local/bin/colmap'
print("Generating mesh from fused point cloud...")
# Ensure we use the full path to the compiled binary
!$COLMAP_BIN poisson_mesher \
    --input_path $DENSE_PATH/fused.ply \
    --output_path $DENSE_PATH/meshed.ply

print("Meshing complete! Final mesh is at /content/dense/meshed.ply")

Generating mesh from fused point cloud...
Value Range: [1.285034,13.494518]
Meshing complete! Final mesh is at /content/dense/meshed.ply


## Exporting Results for Analysis

The resulting 3D models can be viewed locally in software like **MeshLab** or **CloudCompare**. To facilitate this, we archive the three primary components:
1.  **The Database**: Metadata and feature correspondences.
2.  **The Sparse Model**: Camera trajectories and keypoints.
3.  **The Dense Cloud & Mesh**: The high-resolution geometry.

In [47]:
from google.colab import files

# Trigger the download of the combined result package
try:
    print("Requesting download of the full result package...")
    files.download('/content/full_reconstruction_results.zip')
except Exception as e:
    print(f"Download failed. You can find the results in your Google Drive at /COLMAP_Results or download /content/full_reconstruction_results.zip manually from the file pane. Error: {e}")

Requesting download of the full result package...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [43]:
# Package the mesh result specifically for easy download
!zip -j /content/mesh_result.zip /content/dense/meshed.ply

from google.colab import files
try:
    files.download('/content/mesh_result.zip')
    print("Download prompted for mesh_result.zip")
except Exception as e:
    print(f"Manual download required: {e}")

  adding: meshed.ply (deflated 50%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download prompted for mesh_result.zip


## Transfer to Google Drive
The dense.zip file is approximately 1-1.5GB, which is too large to download reliably through the browser session. Instead:


*    Run the cell below to mount your Google Drive (a popup will appear asking for authorization - click "Connect to Google Drive" and follow the prompts to grant permission)
*   After mounting, the code will copy the ZIP files to your Google Drive
*   Navigate to your Google Drive in a browser and download the files from there to your local machine.
*   These files are required on your local machine for visualization purposes

In [44]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [46]:
import os
from google.colab import drive

# 1. Ensure Drive is mounted and directory exists
drive.mount('/content/drive')
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/COLMAP_Results'
!mkdir -p "$DRIVE_OUTPUT_DIR"

# 2. Copy all individual assets to Drive first for safety
print(f"Backing up individual assets to {DRIVE_OUTPUT_DIR}...")
!cp /content/database.db "$DRIVE_OUTPUT_DIR/"
!cp -r /content/sparse "$DRIVE_OUTPUT_DIR/" || echo "Sparse folder not found"
!cp -r /content/dense "$DRIVE_OUTPUT_DIR/" || echo "Dense folder not found"

# 3. Create a single 'Full Results' ZIP for local analysis
print("Packaging all assets into one ZIP for download...")
!zip -r /content/full_reconstruction_results.zip /content/sparse /content/dense /content/database.db

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Backing up individual assets to /content/drive/MyDrive/COLMAP_Results...
Packaging all assets into one ZIP for download...
  adding: content/sparse/ (stored 0%)
  adding: content/sparse/1/ (stored 0%)
  adding: content/sparse/1/images.bin (deflated 70%)
  adding: content/sparse/1/project.ini (deflated 63%)
  adding: content/sparse/1/points3D.bin (deflated 40%)
  adding: content/sparse/1/cameras.bin (deflated 61%)
  adding: content/sparse/0/ (stored 0%)
  adding: content/sparse/0/images.bin (deflated 68%)
  adding: content/sparse/0/project.ini (deflated 63%)
  adding: content/sparse/0/points3D.bin (deflated 40%)
  adding: content/sparse/0/cameras.bin (deflated 61%)
  adding: content/dense/ (stored 0%)
  adding: content/dense/run-colmap-geometric.sh (deflated 60%)
  adding: content/dense/run-colmap-photometric.sh (deflated 60%)
  adding: content/dense/fused.ply